# Ontology And Ask Sage Row-Level Classification Pipeline

This notebook builds the raw CSV ontology, classifies each programming funding row against Genesis Mission Challenges with Ask Sage credentials from `credentials.json`, enriches the graph with evidence edges, and exports the validation tables used by the dashboard.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
import networkx as nx
from IPython.display import display

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from programming_ontology import paths
from programming_ontology.asksage_classifier import (
    AskSageChallengeClassifier,
    default_credentials_path,
    resolve_asksage_openai_base_url,
)
from programming_ontology.build_ontology import build_combined_ontology, export_graph, summarize_graph
from programming_ontology.classify_programming import (
    apply_sme_overrides,
    build_challenge_profiles,
    build_office_claims,
    build_programming_evidence_profiles,
    build_validation_outputs,
    candidate_rankings_to_dataframe,
    classify_programming_records,
    classification_results_to_dataframe,
    enrich_graph_with_reviewed_classifications,
    export_classifications,
    export_validation_outputs,
    load_or_initialize_sme_review,
)

paths.ensure_output_dirs()

In [2]:
combined_graph = build_combined_ontology()
challenges = build_challenge_profiles(combined_graph)
office_claims = build_office_claims(combined_graph, challenges)
claimed_offices = {claim.claimed_office for claim in office_claims}
evidence_rows = build_programming_evidence_profiles(combined_graph, claimed_offices=claimed_offices)

print(summarize_graph(combined_graph))
print(f"Challenge candidates: {len(challenges)}")
print(f"Office claims: {len(office_claims)}")
print(f"Programming evidence rows: {len(evidence_rows)}")

{'name': 'Combined CSV Ontology', 'source_files': 'MSTC_data.csv; programming_data.csv; ProgramOffice_Mapping.csv', 'nodes': 709, 'edges': 2616, 'node_types': {'account_integrator_priority': 1, 'acquisition_end_date': 2, 'acquisition_id': 2, 'acquisition_name': 2, 'acquisition_start_date': 2, 'acquisition_type': 2, 'agency': 1, 'bnr_subtask_code': 31, 'dataset': 3, 'doe_priority_tier': 4, 'expected_outcome': 10, 'fund_value': 3, 'funding_level': 3, 'fy2028_budget': 52, 'fy2029_budget': 49, 'fy2030_budget': 48, 'fy2031_budget': 45, 'fy2032_budget': 44, 'mstc_challenge': 11, 'mstc_record': 10, 'nnsa_advantage': 10, 'planning_scenario': 1, 'problem_statement': 10, 'process_improvement_area': 1, 'program_interest_area': 2, 'program_office_mapping_record': 11, 'program_priority': 16, 'program_request': 18, 'program_value_code': 36, 'programming_record': 61, 'proposed_solution': 10, 'scope_description': 50, 'stat_l3': 9, 'stat_l4': 20, 'stat_l5': 34, 'sub_office': 19, 'submission_type': 1, '

In [3]:
preview = candidate_rankings_to_dataframe(evidence_rows[:10], challenges, top_k=5)
display(preview.head(30))

,programming_record_id,source_row_number,wbs_name,rank,challenge_name,score,matched_terms,challenge_has_mstc_context
0,programming_record:programming_data_csv:1,1,Artificial Intelligence for Nuclear Security (...,1,Accelerating Nuclear Enterprise Acquisition,0.1277,accelerating; advanced; challenges; efficiency...,True
1,programming_record:programming_data_csv:1,1,Artificial Intelligence for Nuclear Security (...,2,Increasing Experimental Capacity at Nuclear Re...,0.0764,advanced; challenges; increasing; nuclear; res...,True
2,programming_record:programming_data_csv:1,1,Artificial Intelligence for Nuclear Security (...,3,"Streamlining Production, Removing Red Tape, an...",0.0693,accelerating; advanced; challenges; efficiency...,True
3,programming_record:programming_data_csv:1,1,Artificial Intelligence for Nuclear Security (...,4,"Accelerating Nuclear Threat Assessment, Prepar...",0.0666,accelerating; intelligence; mission; nuclear; ...,True
4,programming_record:programming_data_csv:1,1,Artificial Intelligence for Nuclear Security (...,5,Accelerating Business and Management Processes...,0.0645,accelerating; efficiency; enterprise; program;...,True
5,programming_record:programming_data_csv:2,2,Artificial Intelligence for Nuclear Security (...,1,Accelerating Nuclear Enterprise Acquisition,0.1266,accelerating; advanced; efficiency; enterprise...,True
6,programming_record:programming_data_csv:2,2,Artificial Intelligence for Nuclear Security (...,2,Accelerating Business and Management Processes...,0.0704,accelerating; efficiency; enterprise; platform...,True
7,programming_record:programming_data_csv:2,2,Artificial Intelligence for Nuclear Security (...,3,Increasing Experimental Capacity at Nuclear Re...,0.0690,advanced; increasing; nuclear; research; secur...,True
8,programming_record:programming_data_csv:2,2,Artificial Intelligence for Nuclear Security (...,4,"Accelerating Nuclear Threat Assessment, Prepar...",0.0669,accelerating; intelligence; mission; nuclear; ...,True
9,programming_record:programming_data_csv:2,2,Artificial Intelligence for Nuclear Security (...,5,"Streamlining Production, Removing Red Tape, an...",0.0638,accelerating; advanced; efficiency; engineerin...,True


In [4]:
ASKSAGE_TOP_K = 3
ASKSAGE_REASONING_EFFORT = os.environ.get("ASKSAGE_REASONING_EFFORT", "medium").strip()
ASKSAGE_RESUME_CACHE = True
ASKSAGE_TIMEOUT_SECONDS = 120
ASKSAGE_CREDENTIALS_PATH = default_credentials_path()
ASKSAGE_MODEL = os.environ.get("ASKSAGE_MODEL", "").strip() or None
ASKSAGE_OPENAI_BASE_URL = resolve_asksage_openai_base_url(credentials_path=ASKSAGE_CREDENTIALS_PATH)

llm_classifier = AskSageChallengeClassifier(
    model=ASKSAGE_MODEL,
    reasoning_effort=ASKSAGE_REASONING_EFFORT,
    credentials_path=ASKSAGE_CREDENTIALS_PATH,
    audit_cache_path=paths.ASKSAGE_CLASSIFICATION_AUDIT_PATH,
    resume_cache=ASKSAGE_RESUME_CACHE,
    timeout_seconds=ASKSAGE_TIMEOUT_SECONDS,
)

print(f"Using Ask Sage credentials file: {ASKSAGE_CREDENTIALS_PATH}")
print(f"Ask Sage model: {llm_classifier.model}")
print(f"Resolved OpenAI SDK base URL: {ASKSAGE_OPENAI_BASE_URL}")
print("Structured output uses credentials.json openai_url when provided; otherwise it derives /openai/v1 from the server URL.")
print("Running one-row structured-output smoke test...")

smoke_results = classify_programming_records(
    evidence_rows[:1],
    challenges,
    llm_classifier=llm_classifier,
    top_k=ASKSAGE_TOP_K,
)
smoke_df = classification_results_to_dataframe(smoke_results)
display(smoke_df[["programming_record_id", "predicted_challenge", "confidence", "rationale"]])

classification_results = classify_programming_records(
    evidence_rows,
    challenges,
    llm_classifier=llm_classifier,
    top_k=ASKSAGE_TOP_K,
)
classification_df = classification_results_to_dataframe(classification_results)
export_classifications(classification_results, paths.PROGRAMMING_CHALLENGE_CLASSIFICATIONS_PATH)
classification_df.to_csv(paths.CLASSIFICATION_DF_PATH, index=False)
display(classification_df.head(10))

Using Ask Sage credentials file: C:\Users\Cameron.Webster\Python\ProgrammingOntology\credentials.json
Ask Sage model: gpt-5.1
Resolved OpenAI SDK base URL: https://api.nnsa.asksage.ai/server/openai/v1
Structured output uses credentials.json openai_url when provided; otherwise it derives /openai/v1 from the server URL.
Running one-row structured-output smoke test...


,programming_record_id,predicted_challenge,confidence,rationale
0,programming_record:programming_data_csv:1,Accelerating Nuclear Enterprise Acquisition,low,The scope description references applying AI t...


,programming_record_id,source_file,source_row_number,wbs_name,wbs_node_id,wbs_code,source_sub_office,normalized_funding_office,stat_l3,stat_l4,...,predicted_challenge,challenge_node_id,challenge_has_mstc_context,confidence,review_flag,rationale,top_candidate_challenges,additional_programming_context,evidence_node_ids,review_reasons
0,programming_record:programming_data_csv:1,programming_data.csv,1,Artificial Intelligence for Nuclear Security (...,wbs:artificial_intelligence_for_nuclear_securi...,OEB.001.003.040.017,NA11,NA11,"Stockpile Research, Technology, and Engineering",Advanced Simulation and Computing,...,Accelerating Nuclear Enterprise Acquisition,mstc_challenge:accelerating_nuclear_enterprise...,True,low,True,The scope description references applying AI t...,Accelerating Nuclear Enterprise Acquisition (0...,Program Int. Area: AI GENESIS MISSION,programming_record:programming_data_csv:1; wbs...,low_confidence
1,programming_record:programming_data_csv:2,programming_data.csv,2,Artificial Intelligence for Nuclear Security (...,wbs:artificial_intelligence_for_nuclear_securi...,OEB.001.003.040.017,NA11,NA11,"Stockpile Research, Technology, and Engineering",Advanced Simulation and Computing,...,Accelerating Business and Management Processes...,mstc_challenge:accelerating_business_and_manag...,True,medium,False,The scope description emphasizes applying AI s...,Accelerating Nuclear Enterprise Acquisition (0...,Program Int. Area: AI GENESIS PLATFORM,programming_record:programming_data_csv:2; wbs...,
2,programming_record:programming_data_csv:3,programming_data.csv,3,Artificial Intelligence for Nuclear Security (...,wbs:artificial_intelligence_for_nuclear_securi...,OEB.001.003.040.017,NA11,NA11,"Stockpile Research, Technology, and Engineering",Advanced Simulation and Computing,...,Harnessing America’s Historic Nuclear Data and...,mstc_challenge:harnessing_america_s_historic_n...,True,medium,False,The scope focuses on productionizing AI tools ...,Integrating Design and Production Operations f...,Program Int. Area: AI GENESIS MISSION; Program...,programming_record:programming_data_csv:3; wbs...,
3,programming_record:programming_data_csv:4,programming_data.csv,4,Artificial Intelligence for Nuclear Security (...,wbs:artificial_intelligence_for_nuclear_securi...,OEB.001.003.040.017,NA11,NA11,"Stockpile Research, Technology, and Engineering",Advanced Simulation and Computing,...,Harnessing America’s Historic Nuclear Data and...,mstc_challenge:harnessing_america_s_historic_n...,True,medium,False,The scope focuses on productionizing AI tools ...,Integrating Design and Production Operations f...,Program Int. Area: AI GENESIS PLATFORM; Progra...,programming_record:programming_data_csv:4; wbs...,
4,programming_record:programming_data_csv:5,programming_data.csv,5,Computational Systems & Software Environment,wbs:computational_systems_software_environment...,OEB.001.003.040.014,NA11,NA11,"Stockpile Research, Technology, and Engineering",Advanced Simulation and Computing,...,Harnessing America’s Historic Nuclear Data and...,mstc_challenge:harnessing_america_s_historic_n...,True,medium,False,The scope focuses on productionizing AI tools ...,Integrating Design and Production Operations f...,Program Int. Area: AI GENESIS PLATFORM; Progra...,programming_record:programming_data_csv:5; wbs...,
5,programming_record:programming_data_csv:6,programming_data.csv,6,Dynamic Materials Properties,wbs:dynamic_materials_properties:afcb5fc411,OEB.001.003.001.002,NA11,NA11,"Stockpile Research, Technology, and Engineering",Assessment Science,...,"Accelerating Materials Discovery, Production, ...",mstc_challenge:accelerating_materials_discover...,True,high,False,The scope description explicitly focuses on 'N...,"Accelerating Materials Discovery, Production, ...",Program Int. Area: AI GENESIS MISSION,programming_record:programming_data_csv:6; wbs...,
6,programming_record:programming_data_csv:7,programming_data.csv,7,Dynamic Materials Properties,wbs:dynamic_materials_

In [5]:
sme_review, created_review_file = load_or_initialize_sme_review(
    paths.SME_REVIEW_OVERRIDES_PATH,
    classification_df,
)
reviewed_classifications = apply_sme_overrides(classification_df, sme_review, challenges)
reviewed_classifications.to_csv(paths.REVIEWED_PROGRAMMING_CHALLENGE_CLASSIFICATIONS_PATH, index=False)

validation_outputs = build_validation_outputs(reviewed_classifications, office_claims)
export_validation_outputs(
    validation_outputs,
    {
        "office_claim_validation": paths.OFFICE_CLAIM_VALIDATION_PATH,
        "challenge_coverage_summary": paths.CHALLENGE_COVERAGE_SUMMARY_PATH,
        "office_challenge_evidence": paths.OFFICE_CHALLENGE_EVIDENCE_PATH,
        "wbs_challenge_summary": paths.WBS_CHALLENGE_SUMMARY_PATH,
    },
)
validation_outputs["challenge_coverage_summary"].to_csv(paths.REVIEW_SUMMARY_PATH, index=False)
print("Created review template." if created_review_file else "Loaded existing review template.")
for name, frame in validation_outputs.items():
    print(f"{name}: {frame.shape[0]} rows x {frame.shape[1]} columns")

Loaded existing review template.
office_claim_validation: 32 rows x 13 columns
challenge_coverage_summary: 11 rows x 11 columns
office_challenge_evidence: 22 rows x 9 columns
wbs_challenge_summary: 46 rows x 10 columns


In [6]:
classified_graph = enrich_graph_with_reviewed_classifications(combined_graph, reviewed_classifications)
export_graph(combined_graph, paths.COMBINED_ONTOLOGY_GRAPH_PATH)
export_graph(classified_graph, paths.CLASSIFIED_ONTOLOGY_GRAPH_PATH)

relation_counts = pd.Series(
    [data.get("relation") for _, _, _, data in classified_graph.edges(keys=True, data=True)]
).value_counts()
display(relation_counts.to_frame("edge_count"))

,edge_count
has_wbs,97
contains_record,82
belongs_to_sub_office,61
has_programming_scenario,61
has_programming_submission_type,61
...,...
wbs_has_programming_acquisition_id,4
wbs_has_programming_acquisition_name,4
wbs_has_programming_acquisition_start_date,4
wbs_has_programming_acquisition_end_date,4
